# Building a first Classification NN

Use the Ames Mutagenicity dataset (from assignment 1A) and build a binary classifier NN. Play with the model parameters. 

For comparison of the NN model performance, consider the performance of other (baseline) classifier models (assignment 1A):
- KNN: Test-Accuracy 0.79, Test-ROC-AUC 0.86
- Decision Tree: Test-Accuracy 0.78, Test-ROC-AUC 0.77
- Random Forest: Test-Accuracy 0.83, Test-ROC-AUC 0.90
- Gradient Boosting: Test-Accuracy 0.77, Test-ROC-AUC 0.85


#### Tasks:
1) Load the dataset `ames_data.csv`. The dataset does not contain any duplicates or NaNs
2) Feature engineering: Calculate various fingerprints from the SMILES strings via mol objects using RDKit(snippet provided for Morgan FPs and MACCS keys)
3) Create feature matrix and target vector. Choose first the MorganFP (Later repeat the process for other fingerprint types). Convert the training and test sets into pytorch tensors.
4) Build your NN (see below for more info)
5) Train your model on the Morgan Fingerprints (and repeat later for other FP types)
6) Evaluate your model's performance and compare to other classifier models
7) Save the model / current state.
8) Respond to the discussion points


#### Note:
The aim of this exercise is to gain a bit of practice in building a simple NN and to see how different parameters and feature engineering influence the model. Maximum accuracy is not the target. 

There is no need to venture too far into the details or more advanced approaches just yet (e.g. batched training would be complete overkill for this assignment - we will discuss that in the next sessions)

0) Import dependencies and datasets

In [12]:
# complete imports if needed for your solution
import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem import MACCSkeys

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

import torch
import torch.nn as nn
import torch.optim as optim

1) Load and investigate the data

In [3]:
df = pd.read_csv("ames_data.csv")
df.head()

,drug_id,smiles,mutagenicity
0,Drug 0,O=[N+]([O-])c1ccc2ccc3ccc([N+](=O)[O-])c4c5ccc...,1
1,Drug 1,O=[N+]([O-])c1c2c(c3ccc4cccc5ccc1c3c45)CCCC2,1
2,Drug 2,O=c1c2ccccc2c(=O)c2c1ccc1c2[nH]c2c3c(=O)c4cccc...,0
3,Drug 3,[N-]=[N+]=CC(=O)NCC(=O)NN,1
4,Drug 4,[N-]=[N+]=C1C=NC(=O)NC1=O,1


2) Generate different fingerprints (try at least one additional FP type as provided in RDKit and use two different fpSizes on one of them) - all of them will be saved in new columns in the Dataframe.

In [5]:
def smiles_to_mol(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return mol

def morganfp(mol):
    fp = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048).GetFingerprint(mol)
    return np.array(fp)

def maccskeys(mol):
    fp = MACCSkeys.GenMACCSKeys(mol)
    return np.array(fp)

def rdkit_fp_1024(mol):
    return np.array(rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=1024).GetFingerprint(mol))

def rdkit_fp_2048(mol):
    return np.array(rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048).GetFingerprint(mol))

fpgens = {
    "MorganFP": morganfp,
    "MACCSkeys": maccskeys,
    "RDKit_1024": rdkit_fp_1024,
    "RDKit_2048": rdkit_fp_2048
}

df["mol"] = df["smiles"].apply(smiles_to_mol)

for name, fpgen in fpgens.items():
    df[name] = df["mol"].apply(fpgen)
df.head()

,drug_id,smiles,mutagenicity,mol,MorganFP,MACCSkeys,RDKit_1024,RDKit_2048
0,Drug 0,O=[N+]([O-])c1ccc2ccc3ccc([N+](=O)[O-])c4c5ccc...,1,<rdkit.Chem.rdchem.Mol object at 0x00000202E52...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, ..."
1,Drug 1,O=[N+]([O-])c1c2c(c3ccc4cccc5ccc1c3c45)CCCC2,1,<rdkit.Chem.rdchem.Mol object at 0x00000202E52...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, ...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, ..."
2,Drug 2,O=c1c2ccccc2c(=O)c2c1ccc1c2[nH]c2c3c(=O)c4cccc...,0,<rdkit.Chem.rdchem.Mol object at 0x00000202E52...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, ...","[1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, ..."
3,Drug 3,[N-]=[N+]=CC(=O)NCC(=O)NN,1,<rdkit.Chem.rdchem.Mol object at 0x00000202E52...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,Drug 4,[N-]=[N+]=C1C=NC(=O)NC1=O,1,<rdkit.Chem.rdchem.Mol object at 0x00000202E52...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, ...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


3. Create feature matrix and target vector. The snippet below converts the data into numpy arrays. Start with the Morgan Fingerprints (and later return here to apply your modell to different fingerprint types - not all of the fingerprints may have the same length, so you may have to adapt the width of your layers).

Do a train startified test split and convert into pytorch tensors.

In [56]:
## MorganFP

X = np.stack(df["MorganFP"].values) # joins multiple arrays along a new axis - builds a proper array from the line-by-line arrays in the df.
y = df["mutagenicity"].to_numpy().astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Convert to Tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [24]:
## RDKit_1024

X = np.stack(df["RDKit_1024"].values) # joins multiple arrays along a new axis - builds a proper array from the line-by-line arrays in the df.
y = df["mutagenicity"].to_numpy().astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Convert to Tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [32]:
## RDKit_2048

X = np.stack(df["RDKit_2048"].values) # joins multiple arrays along a new axis - builds a proper array from the line-by-line arrays in the df.
y = df["mutagenicity"].to_numpy().astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Convert to Tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [36]:
## MACCSkeys

X = np.stack(df["MACCSkeys"].values) # joins multiple arrays along a new axis - builds a proper array from the line-by-line arrays in the df.
y = df["mutagenicity"].to_numpy().astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Convert to Tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

4) Build the NN - adhere to some robust standard values. Start simple and train the model on Morgan FP first.

Optimise the model parameters based on observed over-/underfitting. Experiment with different width and depth, as well as other model parameters. Explore some options to prevent overfitting, e.g. Early stopping (e.g. manually by limiting the epochs) or dropouts. 

Note: Since the input layer needs a lot of neurons (e.g. 2048 bit in the MFPs), consider shrinking the widht from layer to layer. 

Hint: If you use `BCELoss()` as loss function, combine it with a `sigmoid` activation in the last layer. If you use `BCEWithLogitsLoss()`, do not specify any activation in the forward pass (`x = self.outputlayer(x)`).

In [42]:
class BinClassifierNN(nn.Module):
    def __init__(self, input_size):
        super(BinClassifierNN, self).__init__()
        # define your model width and depth below
        self.network = nn.Sequential(
            nn.Linear(input_size, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1) # No Sigmoid here because we use BCEWithLogitsLoss
        )

    def forward(self, x):
        # Specify the forward pass, i.e. activation functions.

        return self.network(x)


In [57]:
# Parameters (change and add as needed)
learning_rate = 0.01
num_epochs = 100

In [50]:
# Parameters (change and add as needed)
learning_rate = 0.000001
num_epochs = 100

In [58]:
input_dim = X_train.shape[1]
model = BinClassifierNN(input_dim)

# choose a loss function for the classification
criterion = nn.BCEWithLogitsLoss()

# choose an optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

5. Train the NN. Note that you may have to squeeze the output (`outputs=models(X_train).squeeze`). This will reduce the actual output of the shape ``[N, 1]`` to ``[N]``, which is comparable to y (The final layer naturally produces a column tensor, which is not directly comparable to the 1D target tensor).

In [59]:
for epoch in range(num_epochs):
    
    model.train()
    outputs = model(X_train).squeeze()
    loss = criterion(outputs, y_train)

    optimizer.zero_grad() # clear existing gradients
    loss.backward() # backpropagation of loss function to optimise gradient
    optimizer.step() # update parameters using Adam

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch + 1}/100], Loss: {loss.item():.5f}')

Epoch [10/100], Loss: 0.51886
Epoch [20/100], Loss: 0.32998
Epoch [30/100], Loss: 0.21997
Epoch [40/100], Loss: 0.13138
Epoch [50/100], Loss: 0.06848
Epoch [60/100], Loss: 0.03287
Epoch [70/100], Loss: 0.01746
Epoch [80/100], Loss: 0.01033
Epoch [90/100], Loss: 0.00693
Epoch [100/100], Loss: 0.00528


6) Evaluate the model. As a first metric, you can use the same loss function to evaluate the model on the test set. For better comparison with methods tested in the assignment 1A (results vide supra), use metrics from scikit-learn (e.g. the accuracy or ROC-AUC score).

Hint: for your prediction you may have to use `squeeze` again to match the target vector in the test set (e.g. ``y_pred = model(X_test).squeeze()``)

In [60]:

model.eval()
with torch.no_grad():
    test_logits = model(X_test).squeeze()
    # Apply sigmoid manually for evaluation metrics
    test_probs = torch.sigmoid(test_logits).numpy()
    test_preds = (test_probs > 0.5).astype(int)

print(f"NN Accuracy: {accuracy_score(y_test, test_preds)}")
print(f"NN ROC-AUC: {roc_auc_score(y_test, test_probs)}")

NN Accuracy: 0.7822802197802198
NN ROC-AUC: 0.8544420023025909


7) Research how you can save the model / current state for later reuse. What are different options here? How can it be loaded again?

In [21]:
# only saving the loaded weights
torch.save(model.state_dict(), 'model_weights.pth')

#to load it at the same time
model.load_state_dict(torch.load('model_weights.pth'))

# or save the entire model => weights and structure
torch.save(model, 'full_model.pth')


#### 8) Discussion points


For comparison of the NN model performance, consider the performance of other (baseline) classifier models (assignment 1A):
- KNN: Test-Accuracy 0.79, Test-ROC-AUC 0.86
- Decision Tree: Test-Accuracy 0.78, Test-ROC-AUC 0.77
- Random Forest: Test-Accuracy 0.83, Test-ROC-AUC 0.90
- Gradient Boosting: Test-Accuracy 0.77, Test-ROC-AUC 0.85


1) How did your model compare to other simple ML classifiers (all used the Morgan FPs)? Discuss!

the NN has accuracy of 0.78 and ROC-AUC of 0.85, so it performed better than the Decision Tree, almost the same as KNN and Gradient boosting but worse compared to Random Forest. 

2) Did you observe any difference between different fingerprint types?

Hint I simply ran the code for the corresponding fingerprint types last before running the NN so I didn't have to make to many copies of the same code. e.g. for RDKit_1024 simply run the cell ## RDKit_1024 and then the other cells of the notebook, and do it the same for the other types. (same holds for parameter sizes etc) 

for RDKit_1024 Accuracy of 0.82, ROC-AUC of 0.88, Loss values of 0.57 decreasing until 0.08
for RDKit_2048 Accuracy of 0.82, ROC-AUC of 0.89, Loss values of 0.56 decreasing until 0.03
for MACCSkeys Accuracy of 0.82, ROC-AUC of 0.89, Loss values of 0.59 decrasing until 0.22


3) Did the fingerprint size impact the model prediction? What message is to be learned from this?

As we see the Accuracy stayed the same when we compare the values of RDKit 1024 and 2048, tho the final loss value of 2024 is lower than for the 1024. Since we know that the accuracy stayed the same, we can assume that the lower loss value of the bigger fingerprint size is likely that it learned more noise. So in this example it is probably not immediately better to have a bigger fingerprint size. This would need to be looked at in individual cases how many bits are necessary to use. Clearly, more features are not always better.

4) What were some model parameters for decent performance depending on the fingerprint type? 

A lower learning rate could be used e.g. from 0.01 to 0.000001. With the RDKit it would be possible to lower the bits from 2048 in steps down until 1 and compare it to force the model to compress the information.


5) Was overfitting a problem? What approaches did you apply to limit that issue? What else would be possible

Yes significantly a problem even. e.g. with Morgan training loss reached a very low number 0.00528 even tho the accuracy was only 0.78 clearly indicating overfitting. The gap between the accuracy and the loss indicates that the model learned the training samples almost by heart rather than learning the rules of the chemistry behind it. 
Possible solutions: Fewer epochs, so stopping earlier helps with overfitting. Dropout is also a good method, such that the model cannot mesmorize the data too easily. 

6) Consider the target "mutagenicity" in the context of molecular structure. 

What does noise mean here?
Mutagenicity data is binary, but the actual biological response is a gradient, so this mismatch introduces noise itself. Also if a molecule counts 0 or 1 could vary in different labs.
And also experimental variabiliy, measurement errors, etc.

How could you use such a predictive model in the lab? 
to pre-screen compounds; too reduce experimental cost; to prioritize safe/ "reliable"/"relevant" molecules

What other data-driven tools could be interesting in this QSAR context?

Graph Neural Networks (GNNs), Molecular Docking, Active learning


7) Why is exporting a full model usually not recommended?

not portable, breaks with changes in the code, hard to debug, version issues etc.